# DuckWADL ADL v3 — Causal Frontier Scored Run

This notebook preserves the **TAAF / The Duck ARC-AGI-3 Kaggle execution path** and installs an in-memory ADL layer before the benchmark begins.

### ADL v3 additions

- **ADL after every real environment move**, including batched model actions.
- **Prediction → action → observed difference → prediction-error update**.
- Persistent per-game causal action statistics and exact-state no-op suppression.
- **Success-Difference Replay**: positive/progressing trajectories are compared with stalled trajectories.
- Analyzer failures and timeouts are **non-fatal**. One local ADL-selected action is executed instead of retrying the same analyzer request indefinitely.
- Difference-compressed model history (8 assistant turns by default).
- A small causal frontier ranks fallback actions by progress probability, information gain, novelty, structural change, risk, loop penalty, and confirmed zero-effect evidence.
- Full artifacts in `/kaggle/working/adl_v3/`:
  - `adl_transitions.jsonl`
  - `adl_analyzer_failures.jsonl`
  - `adl_report.json`
  - `adl_memory.json`

### Required Kaggle inputs

Attach the same three inputs used by the current public Duck harness:

1. `jeroencottaar/taaf-kaggle-source-share`
2. `driessmit1/arc3-vllm-h100-wheelhouse-v3`
3. `driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot`

The competition dataset itself is mounted automatically during a competition rerun.


## 1. Environment and submission mode


In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"

# ADL runtime controls. These are deliberately explicit and may be adjusted in this cell.
os.environ.setdefault("ADL_ANALYZER_HARD_TIMEOUT_S", "35")
os.environ.setdefault("ADL_HISTORY_TURNS", "8")
os.environ.setdefault("ADL_TOOL_STEPS", "6")
os.environ.setdefault("ADL_PROMPT_MAX_CHARS", "3600")
os.environ.setdefault("ADL_NEGATIVE_TRACE_MIN_ACTIONS", "20")
os.environ.setdefault("ADL_FALLBACK_MOUSE_CANDIDATES", "24")
os.environ.setdefault("ADL_STDOUT_EVERY_MOVE", "true")

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry
    for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)]
    if entry
)

WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"DuckWADL: TRUE_SUBMISSION={TRUE_SUBMISSION}")
print(
    "DuckWADL: ADL config=",
    {
        "hard_timeout_s": os.environ["ADL_ANALYZER_HARD_TIMEOUT_S"],
        "history_turns": os.environ["ADL_HISTORY_TURNS"],
        "tool_steps": os.environ["ADL_TOOL_STEPS"],
        "stdout_every_move": os.environ["ADL_STDOUT_EVERY_MOVE"],
    },
)


## 2. Install the official ARC runtime from the offline competition wheelhouse


In [ ]:
wheelhouse = Path(
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
)
if not wheelhouse.is_dir():
    raise RuntimeError(f"ARC-AGI-3 wheelhouse not found: {wheelhouse}")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        str(wheelhouse),
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)
print("DuckWADL: arc-agi runtime installed.")


## 3. Locate the attached TAAF / Duck source bundle


In [ ]:
DATASET_SOURCES = [
    "jeroencottaar/taaf-kaggle-source-share",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
    "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot",
]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"

def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError(
        "TAAF source bundle not found. Attach jeroencottaar/taaf-kaggle-source-share."
    )

def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [
        Path("/kaggle/input") / slug,
        Path("/kaggle/input/datasets") / owner / slug,
    ]

def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]

def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)

BUNDLE_DIR = _find_bundle_dir()
print(f"DuckWADL: source bundle={BUNDLE_DIR}")

kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"DuckWADL: input paths={setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


## 4. Import the bundled source and start the solver's local inference runtime


In [ ]:
def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        raise RuntimeError(f"Bundle source directory missing: {src_root}")
    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries

def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update(
        {str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()}
    )
    return env

source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))

pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"DuckWADL: wrote {pth_path} ({len(source_entries)} source roots)")

env = _command_env()
setup_commands = json.loads((BUNDLE_DIR / "setup_commands.json").read_text())
for command in setup_commands:
    print(f"DuckWADL: setup command: {command}", flush=True)
    subprocess.run(
        command,
        shell=True,
        check=True,
        cwd=WORKING_DIR,
        env=env,
    )
    env = _command_env()
    os.environ.update(env)

for entry in reversed(
    [e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]
):
    if entry not in sys.path:
        sys.path.insert(0, entry)

print("DuckWADL: Duck/TAAF setup complete.")


## 5. Install the complete ADL v3 Causal Frontier layer


In [ ]:
ADL_MODULE_PATH = WORKING_DIR / "duckwadl_adl_v3.py"
ADL_MODULE_SOURCE = 'from __future__ import annotations\n\nimport collections\nimport dataclasses\nimport hashlib\nimport json\nimport math\nimport os\nimport statistics\nimport threading\nimport time\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\nfrom inference.agent.action_names import to_model_action\nfrom inference.agent.runtime_state import Frame, load_runtime_state\nfrom inference.agent.tool_agent import AnalyzerTurnResult, ToolAgent\n\n\n# ---------------------------------------------------------------------------\n# DuckWADL ADL v3 — Causal Frontier\n#\n# Design goals:\n#   1. Every real environment move becomes an ADL transition.\n#   2. Every action is treated as an experiment with a predicted effect.\n#   3. Analyzer timeout/failure is non-fatal: execute one local ADL fallback move.\n#   4. Repeated zero-effect state/action pairs are suppressed.\n#   5. Prompt history is difference-compressed to reduce local-model pressure.\n#   6. Positive trajectories are compared to stalled trajectories online.\n#   7. All ADL events are persisted as JSONL and summarized after the run.\n# ---------------------------------------------------------------------------\n\n\ndef _env_bool(name: str, default: bool) -> bool:\n    raw = os.environ.get(name, "").strip().lower()\n    if not raw:\n        return default\n    if raw in {"1", "true", "yes", "on"}:\n        return True\n    if raw in {"0", "false", "no", "off"}:\n        return False\n    return default\n\n\ndef _env_int(name: str, default: int) -> int:\n    raw = os.environ.get(name, "").strip()\n    if not raw:\n        return default\n    try:\n        return int(raw)\n    except ValueError:\n        return default\n\n\ndef _env_float(name: str, default: float) -> float:\n    raw = os.environ.get(name, "").strip()\n    if not raw:\n        return default\n    try:\n        return float(raw)\n    except ValueError:\n        return default\n\n\nADL_STDOUT_EVERY_MOVE = _env_bool("ADL_STDOUT_EVERY_MOVE", True)\nADL_ANALYZER_HARD_TIMEOUT_S = _env_float("ADL_ANALYZER_HARD_TIMEOUT_S", 35.0)\nADL_HISTORY_TURNS = max(2, _env_int("ADL_HISTORY_TURNS", 8))\nADL_TOOL_STEPS = max(1, _env_int("ADL_TOOL_STEPS", 6))\nADL_PROMPT_MAX_CHARS = max(800, _env_int("ADL_PROMPT_MAX_CHARS", 3600))\nADL_NEGATIVE_TRACE_MIN_ACTIONS = max(5, _env_int("ADL_NEGATIVE_TRACE_MIN_ACTIONS", 20))\nADL_FALLBACK_MOUSE_CANDIDATES = max(4, _env_int("ADL_FALLBACK_MOUSE_CANDIDATES", 24))\n\n_SOURCE = threading.local()\n_PATCH_LOCK = threading.RLock()\n_PATCHED = False\n_ORIGINAL_MAKE_ANALYZER = None\n_ORIGINAL_EXECUTE_ACTION = None\n\n\ndef _set_source(value: str) -> None:\n    _SOURCE.value = value\n\n\ndef _get_source() -> str:\n    return str(getattr(_SOURCE, "value", "model") or "model")\n\n\ndef _grid_shape(grid: tuple[tuple[int, ...], ...]) -> tuple[int, int]:\n    rows = len(grid)\n    cols = max((len(row) for row in grid), default=0)\n    return rows, cols\n\n\ndef _grid_signature(grid: tuple[tuple[int, ...], ...]) -> str:\n    h = hashlib.blake2b(digest_size=10)\n    rows, cols = _grid_shape(grid)\n    h.update(f"{rows}x{cols}|".encode("ascii"))\n    for row in grid:\n        h.update(bytes((int(cell) & 0xFF for cell in row)))\n        h.update(b";")\n    return h.hexdigest()\n\n\ndef _frame_signature(frame: Frame | None) -> str:\n    if frame is None:\n        return "none"\n    return f"L{int(frame.level)}:{_grid_signature(frame.grid)}"\n\n\ndef _histogram(grid: tuple[tuple[int, ...], ...]) -> collections.Counter[int]:\n    c: collections.Counter[int] = collections.Counter()\n    for row in grid:\n        c.update(int(v) for v in row)\n    return c\n\n\ndef _background_color(grid: tuple[tuple[int, ...], ...]) -> int:\n    hist = _histogram(grid)\n    if not hist:\n        return 0\n    return int(hist.most_common(1)[0][0])\n\n\n@dataclasses.dataclass(frozen=True)\nclass GridDelta:\n    changed_cells: int\n    total_cells: int\n    changed_ratio: float\n    bbox: tuple[int, int, int, int] | None\n    transition_counts: tuple[tuple[int, int, int], ...]\n    shape_changed: bool\n\n    def compact(self) -> str:\n        bbox = "none" if self.bbox is None else ",".join(str(v) for v in self.bbox)\n        return (\n            f"cells={self.changed_cells}/{self.total_cells} "\n            f"ratio={self.changed_ratio:.4f} bbox={bbox} "\n            f"shape_changed={int(self.shape_changed)}"\n        )\n\n\ndef compare_frames(before: Frame | None, after: Frame | None) -> GridDelta:\n    if before is None or after is None:\n        return GridDelta(\n            changed_cells=0,\n            total_cells=0,\n            changed_ratio=0.0,\n            bbox=None,\n            transition_counts=(),\n            shape_changed=False,\n        )\n    br, bc = _grid_shape(before.grid)\n    ar, ac = _grid_shape(after.grid)\n    rows = max(br, ar)\n    cols = max(bc, ac)\n    shape_changed = (br, bc) != (ar, ac)\n    changed: list[tuple[int, int]] = []\n    transitions: collections.Counter[tuple[int, int]] = collections.Counter()\n    sentinel = -999\n    for r in range(rows):\n        for c in range(cols):\n            b = before.grid[r][c] if r < br and c < len(before.grid[r]) else sentinel\n            a = after.grid[r][c] if r < ar and c < len(after.grid[r]) else sentinel\n            if b != a:\n                changed.append((r, c))\n                transitions[(int(b), int(a))] += 1\n    total = max(1, rows * cols)\n    if changed:\n        rs = [p[0] for p in changed]\n        cs = [p[1] for p in changed]\n        bbox = (min(rs), min(cs), max(rs), max(cs))\n    else:\n        bbox = None\n    transition_counts = tuple(\n        (int(b), int(a), int(n))\n        for (b, a), n in transitions.most_common(8)\n    )\n    return GridDelta(\n        changed_cells=len(changed),\n        total_cells=total,\n        changed_ratio=float(len(changed)) / float(total),\n        bbox=bbox,\n        transition_counts=transition_counts,\n        shape_changed=shape_changed,\n    )\n\n\ndef _connected_components(\n    grid: tuple[tuple[int, ...], ...],\n    *,\n    background: int | None = None,\n) -> list[dict[str, Any]]:\n    rows, cols = _grid_shape(grid)\n    if rows == 0 or cols == 0:\n        return []\n    bg = _background_color(grid) if background is None else int(background)\n    seen: set[tuple[int, int]] = set()\n    components: list[dict[str, Any]] = []\n    for r in range(rows):\n        row_len = len(grid[r])\n        for c in range(row_len):\n            if (r, c) in seen:\n                continue\n            color = int(grid[r][c])\n            if color == bg:\n                continue\n            stack = [(r, c)]\n            seen.add((r, c))\n            cells: list[tuple[int, int]] = []\n            while stack:\n                rr, cc = stack.pop()\n                if rr >= rows or cc >= len(grid[rr]) or int(grid[rr][cc]) != color:\n                    continue\n                cells.append((rr, cc))\n                for nr, nc in ((rr - 1, cc), (rr + 1, cc), (rr, cc - 1), (rr, cc + 1)):\n                    if nr < 0 or nr >= rows or nc < 0:\n                        continue\n                    if nc >= len(grid[nr]) or (nr, nc) in seen:\n                        continue\n                    if int(grid[nr][nc]) == color:\n                        seen.add((nr, nc))\n                        stack.append((nr, nc))\n            if not cells:\n                continue\n            rs = [x[0] for x in cells]\n            cs = [x[1] for x in cells]\n            r0, r1, c0, c1 = min(rs), max(rs), min(cs), max(cs)\n            center_r = int(round(sum(rs) / len(rs)))\n            center_c = int(round(sum(cs) / len(cs)))\n            components.append(\n                {\n                    "color": color,\n                    "size": len(cells),\n                    "bbox": (r0, c0, r1, c1),\n                    "center": (center_r, center_c),\n                    "density": len(cells) / max(1, (r1 - r0 + 1) * (c1 - c0 + 1)),\n                }\n            )\n    components.sort(key=lambda x: (-int(x["size"]), int(x["color"]), tuple(x["center"])))\n    return components\n\n\ndef _action_name_from_payload(payload: dict[str, Any]) -> str:\n    return str(payload.get("action", "") or "").strip().upper()\n\n\ndef _candidate_key(payload: dict[str, Any]) -> str:\n    action = _action_name_from_payload(payload)\n    if action == "MOUSE":\n        return f"MOUSE@{int(payload.get(\'row\', 0))},{int(payload.get(\'col\', 0))}"\n    return action\n\n\ndef _action_instance_key(action_display: str) -> str:\n    text = str(action_display or "").strip().upper()\n    if text.startswith("MOUSE"):\n        match = __import__("re").search(\n            r"ROW\\s*=\\s*(-?\\d+)\\s*,\\s*COL\\s*=\\s*(-?\\d+)",\n            text,\n        )\n        if match:\n            return f"MOUSE@{int(match.group(1))},{int(match.group(2))}"\n        return "MOUSE"\n    if "(" in text:\n        text = text.split("(", 1)[0].strip()\n    return text\n\n\ndef _action_stat_key(action_display: str) -> str:\n    key = _action_instance_key(action_display)\n    return "MOUSE" if key.startswith("MOUSE") else key\n\n\n@dataclasses.dataclass\nclass ActionStat:\n    trials: int = 0\n    changed: int = 0\n    progress: int = 0\n    game_over: int = 0\n    diff_ratio_sum: float = 0.0\n    prediction_error_sum: float = 0.0\n\n    @property\n    def p_change(self) -> float:\n        return (self.changed + 1.0) / (self.trials + 2.0)\n\n    @property\n    def p_progress(self) -> float:\n        return (self.progress + 0.25) / (self.trials + 5.0)\n\n    @property\n    def p_game_over(self) -> float:\n        return (self.game_over + 0.25) / (self.trials + 5.0)\n\n    @property\n    def expected_diff_ratio(self) -> float:\n        if self.trials <= 0:\n            return 0.02\n        return self.diff_ratio_sum / self.trials\n\n    @property\n    def mean_prediction_error(self) -> float:\n        if self.trials <= 0:\n            return 0.0\n        return self.prediction_error_sum / self.trials\n\n    def update(\n        self,\n        *,\n        changed: bool,\n        progress: bool,\n        game_over: bool,\n        diff_ratio: float,\n        prediction_error: float,\n    ) -> None:\n        self.trials += 1\n        self.changed += int(bool(changed))\n        self.progress += int(bool(progress))\n        self.game_over += int(bool(game_over))\n        self.diff_ratio_sum += float(diff_ratio)\n        self.prediction_error_sum += float(prediction_error)\n\n    def as_dict(self) -> dict[str, Any]:\n        return {\n            "trials": self.trials,\n            "changed": self.changed,\n            "progress": self.progress,\n            "game_over": self.game_over,\n            "p_change": round(self.p_change, 6),\n            "p_progress": round(self.p_progress, 6),\n            "p_game_over": round(self.p_game_over, 6),\n            "expected_diff_ratio": round(self.expected_diff_ratio, 6),\n            "mean_prediction_error": round(self.mean_prediction_error, 6),\n        }\n\n\n@dataclasses.dataclass\nclass TransitionRecord:\n    game_id: str\n    move: int\n    level_before: int\n    level_after: int\n    action: str\n    action_key: str\n    state_before: str\n    state_after: str\n    board_changed: bool\n    diff_cells: int\n    diff_ratio: float\n    diff_bbox: tuple[int, int, int, int] | None\n    progress_delta: int\n    reward: float\n    game_over: bool\n    run_complete: bool\n    source: str\n    predicted_change: float\n    predicted_progress: float\n    predicted_diff_ratio: float\n    prediction_error: float\n    timestamp: float\n\n    def as_dict(self) -> dict[str, Any]:\n        return dataclasses.asdict(self)\n\n\n@dataclasses.dataclass\nclass AnalyzerFailure:\n    game_id: str\n    move: int\n    reason: str\n    latency_s: float\n    timestamp: float\n\n    def as_dict(self) -> dict[str, Any]:\n        return dataclasses.asdict(self)\n\n\n@dataclasses.dataclass\nclass Candidate:\n    payload: dict[str, Any]\n    key: str\n    stat_key: str\n    score: float\n    terms: dict[str, float]\n\n    def compact(self) -> str:\n        return (\n            f"{self.key} score={self.score:.3f} "\n            + " ".join(f"{k}={v:.2f}" for k, v in sorted(self.terms.items()))\n        )\n\n\n@dataclasses.dataclass\nclass GameTrace:\n    game_id: str\n    transitions: list[TransitionRecord] = dataclasses.field(default_factory=list)\n    final_score: float | None = None\n\n    @property\n    def progress_events(self) -> int:\n        return sum(1 for t in self.transitions if t.progress_delta > 0)\n\n    def features(self) -> dict[str, float]:\n        ts = self.transitions\n        n = len(ts)\n        if n == 0:\n            return {\n                "actions": 0.0,\n                "change_rate": 0.0,\n                "zero_effect_rate": 0.0,\n                "progress_rate": 0.0,\n                "unique_state_rate": 0.0,\n                "repeat_action_rate": 0.0,\n                "fallback_rate": 0.0,\n                "mean_diff_ratio": 0.0,\n                "mean_prediction_error": 0.0,\n            }\n        changed = sum(int(t.board_changed) for t in ts)\n        progress = sum(int(t.progress_delta > 0) for t in ts)\n        unique_states = len({t.state_after for t in ts})\n        repeats = sum(\n            1\n            for i in range(1, n)\n            if ts[i].action_key == ts[i - 1].action_key\n            and ts[i].state_before == ts[i - 1].state_before\n        )\n        fallback = sum(int(t.source == "fallback") for t in ts)\n        return {\n            "actions": float(n),\n            "change_rate": changed / n,\n            "zero_effect_rate": (n - changed) / n,\n            "progress_rate": progress / n,\n            "unique_state_rate": unique_states / n,\n            "repeat_action_rate": repeats / max(1, n - 1),\n            "fallback_rate": fallback / n,\n            "mean_diff_ratio": sum(t.diff_ratio for t in ts) / n,\n            "mean_prediction_error": sum(t.prediction_error for t in ts) / n,\n        }\n\n\ndef _default_adl_output_dir() -> Path:\n    explicit = os.environ.get("ADL_OUTPUT_DIR", "").strip()\n    if explicit:\n        return Path(explicit)\n    kaggle_working = Path("/kaggle/working")\n    if kaggle_working.is_dir():\n        return kaggle_working / "adl_v3"\n    return Path("/tmp/duckwadl_adl_v3")\n\n\nclass ADLRunMemory:\n    def __init__(self, output_dir: str | Path | None = None) -> None:\n        self.output_dir = Path(output_dir) if output_dir is not None else _default_adl_output_dir()\n        self.output_dir.mkdir(parents=True, exist_ok=True)\n        self.events_path = self.output_dir / "adl_transitions.jsonl"\n        self.failures_path = self.output_dir / "adl_analyzer_failures.jsonl"\n        self.report_path = self.output_dir / "adl_report.json"\n        self.memory_path = self.output_dir / "adl_memory.json"\n        self._lock = threading.RLock()\n        self.traces: dict[str, GameTrace] = {}\n        self.action_stats: dict[tuple[str, str], ActionStat] = {}\n        self.state_attempts: collections.Counter[tuple[str, str, str]] = collections.Counter()\n        self.state_zero_effect: collections.Counter[tuple[str, str, str]] = collections.Counter()\n        self.analyzer_failures: list[AnalyzerFailure] = []\n        self._recent_state_order: dict[str, collections.deque[str]] = {}\n        self._recent_action_order: dict[str, collections.deque[str]] = {}\n\n    def reset_files(self) -> None:\n        with self._lock:\n            self.events_path.parent.mkdir(parents=True, exist_ok=True)\n            self.events_path.write_text("", encoding="utf-8")\n            self.failures_path.write_text("", encoding="utf-8")\n\n    def _trace(self, game_id: str) -> GameTrace:\n        game_id = str(game_id or "unknown")\n        if game_id not in self.traces:\n            self.traces[game_id] = GameTrace(game_id=game_id)\n        return self.traces[game_id]\n\n    def _stat(self, game_id: str, stat_key: str) -> ActionStat:\n        key = (str(game_id), str(stat_key))\n        if key not in self.action_stats:\n            self.action_stats[key] = ActionStat()\n        return self.action_stats[key]\n\n    def prediction(self, game_id: str, action_display: str) -> tuple[float, float, float]:\n        with self._lock:\n            stat = self._stat(game_id, _action_stat_key(action_display))\n            return stat.p_change, stat.p_progress, stat.expected_diff_ratio\n\n    def record_transition(\n        self,\n        *,\n        game_id: str,\n        move: int,\n        before: Frame,\n        after: Frame,\n        action_display: str,\n        payload: dict[str, Any],\n        source: str,\n    ) -> TransitionRecord:\n        delta = compare_frames(before, after)\n        state_before = _frame_signature(before)\n        state_after = _frame_signature(after)\n        action_display_text = str(\n            action_display or payload.get("action_display") or payload.get("action_name") or ""\n        ).strip()\n        action_instance_key = _action_instance_key(action_display_text)\n        stat_key = _action_stat_key(action_display_text)\n        level_before = int(getattr(before, "level", 1) or 1)\n        level_after = int(getattr(after, "level", level_before) or level_before)\n        progress_delta = max(\n            int(level_after > level_before),\n            int(bool(payload.get("level_completed"))),\n            int(bool(payload.get("run_complete"))),\n        )\n        reward = float(payload.get("reward", 0.0) or 0.0)\n        if reward > 0:\n            progress_delta = max(progress_delta, 1)\n\n        with self._lock:\n            stat = self._stat(game_id, stat_key)\n            predicted_change = stat.p_change\n            predicted_progress = stat.p_progress\n            predicted_diff_ratio = stat.expected_diff_ratio\n            actual_change = 1.0 if delta.changed_cells > 0 else 0.0\n            actual_progress = 1.0 if progress_delta > 0 else 0.0\n            diff_scale = max(0.01, predicted_diff_ratio, delta.changed_ratio)\n            prediction_error = (\n                abs(predicted_change - actual_change)\n                + abs(predicted_progress - actual_progress)\n                + min(1.0, abs(predicted_diff_ratio - delta.changed_ratio) / diff_scale)\n            ) / 3.0\n\n            record = TransitionRecord(\n                game_id=str(game_id),\n                move=int(move),\n                level_before=level_before,\n                level_after=level_after,\n                action=action_display_text,\n                action_key=action_instance_key,\n                state_before=state_before,\n                state_after=state_after,\n                board_changed=delta.changed_cells > 0,\n                diff_cells=delta.changed_cells,\n                diff_ratio=delta.changed_ratio,\n                diff_bbox=delta.bbox,\n                progress_delta=progress_delta,\n                reward=reward,\n                game_over=bool(payload.get("game_over")),\n                run_complete=bool(payload.get("run_complete")),\n                source=str(source or "model"),\n                predicted_change=predicted_change,\n                predicted_progress=predicted_progress,\n                predicted_diff_ratio=predicted_diff_ratio,\n                prediction_error=prediction_error,\n                timestamp=time.time(),\n            )\n            self._trace(game_id).transitions.append(record)\n            self.state_attempts[(str(game_id), state_before, action_instance_key)] += 1\n            if not record.board_changed and record.progress_delta <= 0:\n                self.state_zero_effect[(str(game_id), state_before, action_instance_key)] += 1\n            stat.update(\n                changed=record.board_changed,\n                progress=record.progress_delta > 0,\n                game_over=record.game_over,\n                diff_ratio=record.diff_ratio,\n                prediction_error=record.prediction_error,\n            )\n            self._recent_state_order.setdefault(\n                str(game_id), collections.deque(maxlen=16)\n            ).append(state_after)\n            self._recent_action_order.setdefault(\n                str(game_id), collections.deque(maxlen=16)\n            ).append(action_instance_key)\n            with self.events_path.open("a", encoding="utf-8") as f:\n                f.write(json.dumps(record.as_dict(), sort_keys=True) + "\\n")\n            return record\n\n    def record_failure(self, game_id: str, move: int, reason: str, latency_s: float) -> None:\n        failure = AnalyzerFailure(\n            game_id=str(game_id),\n            move=int(move),\n            reason=str(reason),\n            latency_s=float(latency_s),\n            timestamp=time.time(),\n        )\n        with self._lock:\n            self.analyzer_failures.append(failure)\n            with self.failures_path.open("a", encoding="utf-8") as f:\n                f.write(json.dumps(failure.as_dict(), sort_keys=True) + "\\n")\n\n    def set_final_score(self, game_id: str, score: float | None) -> None:\n        with self._lock:\n            trace = self._trace(game_id)\n            trace.final_score = None if score is None else float(score)\n\n    def _feature_prototypes(self) -> tuple[dict[str, float], dict[str, float], dict[str, float]]:\n        positives: list[dict[str, float]] = []\n        negatives: list[dict[str, float]] = []\n        for trace in self.traces.values():\n            features = trace.features()\n            positive = (\n                trace.progress_events > 0\n                or (trace.final_score is not None and trace.final_score > 0)\n            )\n            if positive:\n                positives.append(features)\n            elif len(trace.transitions) >= ADL_NEGATIVE_TRACE_MIN_ACTIONS:\n                negatives.append(features)\n\n        keys = [\n            "change_rate",\n            "zero_effect_rate",\n            "progress_rate",\n            "unique_state_rate",\n            "repeat_action_rate",\n            "fallback_rate",\n            "mean_diff_ratio",\n            "mean_prediction_error",\n        ]\n\n        def mean_feature(rows: list[dict[str, float]]) -> dict[str, float]:\n            if not rows:\n                return {k: 0.0 for k in keys}\n            return {k: sum(row.get(k, 0.0) for row in rows) / len(rows) for k in keys}\n\n        pos = mean_feature(positives)\n        neg = mean_feature(negatives)\n        delta = {k: pos[k] - neg[k] for k in keys}\n        return pos, neg, delta\n\n    def strategy_delta(self) -> dict[str, float]:\n        with self._lock:\n            return self._feature_prototypes()[2]\n\n    def global_guidance(self) -> list[str]:\n        with self._lock:\n            positive_count = sum(\n                1\n                for trace in self.traces.values()\n                if trace.progress_events > 0\n                or (trace.final_score is not None and trace.final_score > 0)\n            )\n            negative_count = sum(\n                1\n                for trace in self.traces.values()\n                if trace.progress_events <= 0\n                and not (trace.final_score is not None and trace.final_score > 0)\n                and len(trace.transitions) >= ADL_NEGATIVE_TRACE_MIN_ACTIONS\n            )\n            if positive_count == 0:\n                return [\n                    "No positive trajectory prototype yet; maximize information gain and avoid repeating zero-effect state/action pairs."\n                ]\n            if negative_count == 0:\n                return [\n                    "Positive trajectory evidence exists; continue favoring unique state changes and confirmed progress-producing actions."\n                ]\n            _, _, d = self._feature_prototypes()\n            guidance: list[str] = []\n            if d["change_rate"] > 0.05:\n                guidance.append("Successful traces produce board changes more often; prefer actions with higher learned change probability.")\n            if d["unique_state_rate"] > 0.05:\n                guidance.append("Successful traces visit more unique states; increase novelty pressure when stalled.")\n            if d["zero_effect_rate"] < -0.05:\n                guidance.append("Successful traces waste fewer moves on zero-effect actions; suppress confirmed no-op state/action pairs.")\n            if d["repeat_action_rate"] < -0.03:\n                guidance.append("Successful traces repeat the same state/action less; penalize loops and immediate retries.")\n            if d["mean_diff_ratio"] > 0.005:\n                guidance.append("Successful traces create larger structural deltas; prefer meaningful transformations over tiny/no changes.")\n            if not guidance:\n                guidance.append("Success/failure prototypes are currently close; rely on local causal evidence and uncertainty-driven tests.")\n            return guidance[:4]\n\n    def _weights_for_game(self, game_id: str) -> dict[str, float]:\n        trace = self._trace(game_id)\n        actions = len(trace.transitions)\n        has_progress = trace.progress_events > 0\n        if not has_progress and actions < 30:\n            weights = {\n                "progress": 4.0,\n                "change": 1.8,\n                "info": 2.4,\n                "novelty": 2.8,\n                "diff": 1.0,\n                "risk": 3.0,\n                "repeat": 2.8,\n                "zero": 4.5,\n            }\n        elif not has_progress:\n            weights = {\n                "progress": 4.5,\n                "change": 2.2,\n                "info": 1.8,\n                "novelty": 3.2,\n                "diff": 1.3,\n                "risk": 3.2,\n                "repeat": 3.5,\n                "zero": 5.0,\n            }\n        else:\n            weights = {\n                "progress": 6.0,\n                "change": 2.0,\n                "info": 1.0,\n                "novelty": 1.4,\n                "diff": 1.4,\n                "risk": 4.0,\n                "repeat": 2.4,\n                "zero": 4.0,\n            }\n        d = self._feature_prototypes()[2]\n        if d["change_rate"] > 0.05:\n            weights["change"] += min(1.5, d["change_rate"] * 4.0)\n        if d["unique_state_rate"] > 0.05:\n            weights["novelty"] += min(1.5, d["unique_state_rate"] * 4.0)\n        if d["repeat_action_rate"] < -0.03:\n            weights["repeat"] += min(1.5, abs(d["repeat_action_rate"]) * 4.0)\n        if d["zero_effect_rate"] < -0.05:\n            weights["zero"] += min(1.5, abs(d["zero_effect_rate"]) * 4.0)\n        return weights\n\n    def _mouse_candidates(self, frame: Frame) -> list[dict[str, Any]]:\n        grid = frame.grid\n        rows, cols = _grid_shape(grid)\n        if rows == 0 or cols == 0:\n            return [{"action": "MOUSE", "row": 0, "col": 0}]\n        components = _connected_components(grid)\n        points: list[tuple[int, int]] = []\n        for comp in components[:ADL_FALLBACK_MOUSE_CANDIDATES]:\n            center = tuple(comp["center"])\n            points.append((int(center[0]), int(center[1])))\n            r0, c0, r1, c1 = tuple(comp["bbox"])\n            if int(comp["size"]) >= 4:\n                points.extend(\n                    [\n                        ((r0 + r1) // 2, c0),\n                        ((r0 + r1) // 2, c1),\n                        (r0, (c0 + c1) // 2),\n                        (r1, (c0 + c1) // 2),\n                    ]\n                )\n        # Always include geometric probes in case the meaningful target shares the background color.\n        points.extend(\n            [\n                (rows // 2, cols // 2),\n                (rows // 4, cols // 4),\n                (rows // 4, max(0, (3 * cols) // 4)),\n                (max(0, (3 * rows) // 4), cols // 4),\n                (max(0, (3 * rows) // 4), max(0, (3 * cols) // 4)),\n            ]\n        )\n        unique: list[dict[str, Any]] = []\n        seen: set[tuple[int, int]] = set()\n        for r, c in points:\n            r = max(0, min(rows - 1, int(r)))\n            c = max(0, min(max(0, len(grid[r]) - 1), int(c)))\n            if (r, c) in seen:\n                continue\n            seen.add((r, c))\n            unique.append({"action": "MOUSE", "row": r, "col": c})\n            if len(unique) >= ADL_FALLBACK_MOUSE_CANDIDATES:\n                break\n        return unique\n\n    def generate_candidates(\n        self,\n        game_id: str,\n        frame: Frame,\n        valid_actions: Iterable[str],\n    ) -> list[Candidate]:\n        normalized: list[str] = []\n        for raw in valid_actions:\n            name = to_model_action(str(raw)).strip().upper()\n            if name and name != "RESET" and name not in normalized:\n                normalized.append(name)\n        if not normalized:\n            return []\n\n        payloads: list[dict[str, Any]] = []\n        for action in normalized:\n            if action == "MOUSE":\n                payloads.extend(self._mouse_candidates(frame))\n            else:\n                payloads.append({"action": action})\n\n        state_sig = _frame_signature(frame)\n        weights = self._weights_for_game(game_id)\n        recent_states = list(self._recent_state_order.get(str(game_id), ()))\n        recent_actions = list(self._recent_action_order.get(str(game_id), ()))\n        result: list[Candidate] = []\n\n        with self._lock:\n            for payload in payloads:\n                key = _candidate_key(payload)\n                stat_key = _action_stat_key(key)\n                stat = self._stat(game_id, stat_key)\n                state_trials = self.state_attempts[(str(game_id), state_sig, key)]\n                zero_count = self.state_zero_effect[(str(game_id), state_sig, key)]\n                novelty = 1.0 / (1.0 + state_trials)\n                info = 1.0 / math.sqrt(1.0 + stat.trials)\n                # Bernoulli entropy is maximal at p=0.5, adding useful causal-test pressure.\n                p = min(1.0 - 1e-9, max(1e-9, stat.p_change))\n                entropy = -(p * math.log2(p) + (1.0 - p) * math.log2(1.0 - p))\n                info = 0.55 * info + 0.45 * entropy\n                repeat_penalty = 0.0\n                if recent_actions and key == recent_actions[-1]:\n                    repeat_penalty += 0.7\n                if len(recent_actions) >= 3 and all(key == x for x in recent_actions[-3:]):\n                    repeat_penalty += 0.8\n                if recent_states.count(state_sig) >= 3:\n                    repeat_penalty += min(1.0, 0.15 * recent_states.count(state_sig))\n                zero_penalty = min(1.0, float(zero_count))\n                score_terms = {\n                    "progress": weights["progress"] * stat.p_progress,\n                    "change": weights["change"] * stat.p_change,\n                    "info": weights["info"] * info,\n                    "novelty": weights["novelty"] * novelty,\n                    "diff": weights["diff"] * min(1.0, stat.expected_diff_ratio * 10.0),\n                    "risk": -weights["risk"] * stat.p_game_over,\n                    "repeat": -weights["repeat"] * repeat_penalty,\n                    "zero": -weights["zero"] * zero_penalty,\n                }\n                score = sum(score_terms.values())\n                # Deterministic tie-breaker derived from state + candidate.\n                jitter_seed = hashlib.blake2b(\n                    f"{state_sig}|{key}".encode("utf-8"), digest_size=4\n                ).digest()\n                jitter = int.from_bytes(jitter_seed, "big") / 2**32\n                score += jitter * 1e-4\n                result.append(\n                    Candidate(\n                        payload=payload,\n                        key=key,\n                        stat_key=stat_key,\n                        score=score,\n                        terms=score_terms,\n                    )\n                )\n        result.sort(key=lambda c: (-c.score, c.key))\n        return result\n\n    def choose_fallback(\n        self,\n        game_id: str,\n        frame: Frame,\n        valid_actions: Iterable[str],\n    ) -> tuple[Candidate | None, list[Candidate]]:\n        candidates = self.generate_candidates(game_id, frame, valid_actions)\n        return (candidates[0] if candidates else None), candidates[:4]\n\n    def prompt_context(self, game_id: str, current_frame: Frame | None) -> str:\n        if current_frame is None:\n            return ""\n        with self._lock:\n            trace = self._trace(game_id)\n            state_sig = _frame_signature(current_frame)\n            recent = trace.transitions[-6:]\n            action_rows: list[tuple[str, ActionStat]] = [\n                (stat_key, stat)\n                for (gid, stat_key), stat in self.action_stats.items()\n                if gid == str(game_id)\n            ]\n            action_rows.sort(key=lambda item: (-item[1].trials, item[0]))\n            negatives = [\n                action_key\n                for (gid, sig, action_key), count in self.state_zero_effect.items()\n                if gid == str(game_id) and sig == state_sig and count > 0\n            ]\n            guidance = self.global_guidance()\n            lines = [\n                "",\n                "[ADL CAUSAL DIFFERENCE MEMORY]",\n                "Treat the next action as an experiment: predict its effect, execute, observe the delta, then revise the causal model.",\n                f"Current ADL phase: {\'exploit/validate\' if trace.progress_events else (\'stalled-explore\' if len(trace.transitions) >= 30 else \'explore\')}.",\n                f"Recorded moves this game: {len(trace.transitions)}; progress events: {trace.progress_events}.",\n            ]\n            if recent:\n                lines.append("Recent observed differences:")\n                for t in recent:\n                    lines.append(\n                        f"- move {t.move}: {t.action} -> changed={int(t.board_changed)} "\n                        f"delta={t.diff_cells} ({t.diff_ratio:.3f}) progress={t.progress_delta} "\n                        f"pred_error={t.prediction_error:.3f} source={t.source}"\n                    )\n            if action_rows:\n                lines.append("Learned action-effect estimates:")\n                for name, stat in action_rows[:7]:\n                    lines.append(\n                        f"- {name}: n={stat.trials} P(change)={stat.p_change:.2f} "\n                        f"P(progress)={stat.p_progress:.2f} P(game_over)={stat.p_game_over:.2f} "\n                        f"E(diff)={stat.expected_diff_ratio:.3f}"\n                    )\n            if negatives:\n                lines.append(\n                    "Confirmed zero-effect actions in this exact state: "\n                    + ", ".join(sorted(set(negatives))[:10])\n                )\n            lines.append("Cross-game Success-Difference Replay:")\n            for item in guidance:\n                lines.append(f"- {item}")\n            lines.append(\n                "ADL execution rule: prefer one information-rich action at a time. "\n                "Use a batch only after the intermediate effects are already causally confirmed."\n            )\n            lines.append("[END ADL MEMORY]")\n            rendered = "\\n".join(lines)\n            if len(rendered) > ADL_PROMPT_MAX_CHARS:\n                rendered = rendered[-ADL_PROMPT_MAX_CHARS:]\n                rendered = "[ADL MEMORY TRIMMED]\\n" + rendered\n            return rendered\n\n    def finalize_from_benchmark(self, benchmark: Any) -> dict[str, Any]:\n        with self._lock:\n            for game in list(getattr(benchmark, "games", []) or []):\n                run = getattr(game, "game_run", None)\n                if run is None:\n                    continue\n                game_id = str(getattr(run, "game_id", "") or getattr(game, "env_name", "") or "unknown")\n                final_score = getattr(run, "final_score", None)\n                if final_score is not None:\n                    self.set_final_score(game_id, float(final_score))\n            report = self.report()\n            self.report_path.write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")\n            memory = {\n                "action_stats": {\n                    f"{gid}|{action}": stat.as_dict()\n                    for (gid, action), stat in sorted(self.action_stats.items())\n                },\n                "strategy_delta": self.strategy_delta(),\n                "guidance": self.global_guidance(),\n            }\n            self.memory_path.write_text(json.dumps(memory, indent=2, sort_keys=True), encoding="utf-8")\n            return report\n\n    def _nearest_failure_delta(self, positive: GameTrace, negatives: list[GameTrace]) -> dict[str, Any] | None:\n        if not negatives:\n            return None\n        pf = positive.features()\n        nearest = min(\n            negatives,\n            key=lambda trace: abs(len(trace.transitions) - len(positive.transitions)),\n        )\n        nf = nearest.features()\n        keys = [\n            "change_rate",\n            "zero_effect_rate",\n            "unique_state_rate",\n            "repeat_action_rate",\n            "fallback_rate",\n            "mean_diff_ratio",\n            "mean_prediction_error",\n        ]\n        return {\n            "positive_game": positive.game_id,\n            "negative_game": nearest.game_id,\n            "positive_actions": len(positive.transitions),\n            "negative_actions": len(nearest.transitions),\n            "delta": {k: round(pf[k] - nf[k], 6) for k in keys},\n        }\n\n    def report(self) -> dict[str, Any]:\n        with self._lock:\n            pos_proto, neg_proto, delta = self._feature_prototypes()\n            positives = [\n                trace\n                for trace in self.traces.values()\n                if trace.progress_events > 0\n                or (trace.final_score is not None and trace.final_score > 0)\n            ]\n            negatives = [\n                trace\n                for trace in self.traces.values()\n                if trace.progress_events <= 0\n                and not (trace.final_score is not None and trace.final_score > 0)\n                and len(trace.transitions) >= ADL_NEGATIVE_TRACE_MIN_ACTIONS\n            ]\n            replay = [\n                item\n                for trace in positives\n                for item in [self._nearest_failure_delta(trace, negatives)]\n                if item is not None\n            ]\n            per_game = {}\n            for gid, trace in sorted(self.traces.items()):\n                per_game[gid] = {\n                    "moves": len(trace.transitions),\n                    "progress_events": trace.progress_events,\n                    "final_score": trace.final_score,\n                    "features": {k: round(v, 6) for k, v in trace.features().items()},\n                    "fallback_moves": sum(int(t.source == "fallback") for t in trace.transitions),\n                    "analyzer_failures": sum(1 for f in self.analyzer_failures if f.game_id == gid),\n                }\n            return {\n                "version": "DuckWADL-ADL-v3-Causal-Frontier",\n                "generated_at_epoch": time.time(),\n                "totals": {\n                    "games_seen": len(self.traces),\n                    "moves_recorded": sum(len(t.transitions) for t in self.traces.values()),\n                    "positive_traces": len(positives),\n                    "stalled_negative_traces": len(negatives),\n                    "analyzer_failures": len(self.analyzer_failures),\n                    "fallback_moves": sum(\n                        int(t.source == "fallback")\n                        for trace in self.traces.values()\n                        for t in trace.transitions\n                    ),\n                },\n                "positive_prototype": {k: round(v, 6) for k, v in pos_proto.items()},\n                "negative_prototype": {k: round(v, 6) for k, v in neg_proto.items()},\n                "strategy_delta": {k: round(v, 6) for k, v in delta.items()},\n                "guidance": self.global_guidance(),\n                "success_difference_replay": replay,\n                "games": per_game,\n            }\n\n\nADL_SHARED = ADLRunMemory()\n\n\nclass ADLToolAgent(ToolAgent):\n    """Duck ToolAgent with difference compression and a non-fatal local fallback."""\n\n    def __init__(\n        self,\n        *args: Any,\n        game_id: str = "unknown",\n        shared_memory: ADLRunMemory | None = None,\n        hard_timeout_s: float = ADL_ANALYZER_HARD_TIMEOUT_S,\n        history_turns: int = ADL_HISTORY_TURNS,\n        tool_steps: int = ADL_TOOL_STEPS,\n        **kwargs: Any,\n    ) -> None:\n        super().__init__(*args, **kwargs)\n        self.game_id = str(game_id or "unknown")\n        self.shared_memory = shared_memory or ADL_SHARED\n        self.hard_timeout_s = max(1.0, float(hard_timeout_s))\n        self.history_turns = max(2, int(history_turns))\n        self._tool_steps = max(1, int(tool_steps))\n        self._last_request_error = ""\n        # Reduce persistent prompt pressure. The causal summary is carried separately.\n        self._context_budget_tokens = min(int(self._context_budget_tokens), 24576)\n\n    def _chat_completion(self, *args: Any, **kwargs: Any) -> Any:\n        try:\n            return super()._chat_completion(*args, **kwargs)\n        except Exception as exc:\n            self._last_request_error = f"{type(exc).__name__}: {exc}"[:700]\n            raise\n\n    def _persistent_history_messages(\n        self,\n        messages: list[dict[str, Any]],\n        *,\n        tools: list[dict[str, Any]] | None = None,\n    ) -> list[dict[str, Any]]:\n        trimmed = self._trim_messages_for_context(messages, tools=tools)\n        if not trimmed:\n            return []\n        trimmed_history = trimmed[1:]\n        history = self._keep_recent_history_turns(\n            trimmed_history,\n            max_turns=self.history_turns,\n        )\n        if (\n            history\n            and str(history[0].get("role", "")).strip() != "user"\n            and len(trimmed_history) > len(history)\n        ):\n            previous_message = trimmed_history[len(trimmed_history) - len(history) - 1]\n            if str(previous_message.get("role", "")).strip() == "user":\n                history = [previous_message, *history]\n        return self._drop_until_first_user_message(history)\n\n    def _build_user_prompt(\n        self,\n        action_num: int,\n        *,\n        valid_actions: list[str] | None,\n        current_frame: Frame | None = None,\n        history_entries: list[Any] | None = None,\n        previous_step_summary: dict[str, Any] | None = None,\n    ) -> str:\n        base = super()._build_user_prompt(\n            action_num,\n            valid_actions=valid_actions,\n            current_frame=current_frame,\n            history_entries=history_entries,\n            previous_step_summary=previous_step_summary,\n        )\n        return base + "\\n" + self.shared_memory.prompt_context(self.game_id, current_frame)\n\n    def _effective_request_timeout(self, requested: float | None) -> float:\n        if requested is None:\n            return self.hard_timeout_s\n        try:\n            parsed = float(requested)\n        except (TypeError, ValueError):\n            return self.hard_timeout_s\n        return max(0.1, min(parsed, self.hard_timeout_s))\n\n    def _append_adl_status(self, transcript_path: Path | None, text: str) -> None:\n        if transcript_path is None:\n            return\n        try:\n            transcript_path.parent.mkdir(parents=True, exist_ok=True)\n            with transcript_path.open("a", encoding="utf-8") as f:\n                f.write("\\n[ADL STATUS]\\n")\n                f.write(text.rstrip())\n                f.write("\\n\\n")\n        except OSError:\n            pass\n\n    def _fallback(\n        self,\n        *,\n        state_path: Path,\n        action_num: int,\n        valid_actions: list[str] | None,\n        step_env: Callable[[dict[str, Any]], dict[str, Any]] | None,\n        transcript_path: Path | None,\n        reason: str,\n        latency_s: float,\n        should_stop: Callable[[], bool] | None,\n    ) -> AnalyzerTurnResult:\n        self.shared_memory.record_failure(self.game_id, action_num, reason, latency_s)\n        if should_stop is not None:\n            try:\n                if should_stop():\n                    return AnalyzerTurnResult(\n                        step_executed=False,\n                        retryable_failure=False,\n                        reasoning=f"ADL fallback skipped because solver is stopping: {reason}",\n                        yielded_control=True,\n                    )\n            except Exception:\n                pass\n        if step_env is None:\n            return AnalyzerTurnResult(\n                step_executed=False,\n                retryable_failure=False,\n                reasoning=f"ADL fallback unavailable: no step_env callback ({reason})",\n            )\n\n        frame, _ = load_runtime_state(state_path)\n        if frame is None:\n            return AnalyzerTurnResult(\n                step_executed=False,\n                retryable_failure=False,\n                reasoning=f"ADL fallback unavailable: runtime frame missing ({reason})",\n            )\n\n        choice, frontier = self.shared_memory.choose_fallback(\n            self.game_id,\n            frame,\n            valid_actions or [],\n        )\n        frontier_text = " | ".join(c.compact() for c in frontier)\n        if choice is None:\n            self._append_adl_status(\n                transcript_path,\n                f"fallback_reason={reason}\\nNo valid non-RESET fallback action.\\nfrontier={frontier_text}",\n            )\n            return AnalyzerTurnResult(\n                step_executed=False,\n                retryable_failure=False,\n                reasoning=f"ADL fallback found no valid non-RESET action ({reason})",\n            )\n\n        previous_source = _get_source()\n        try:\n            _set_source("fallback")\n            payload = step_env({"actions": [choice.payload]})\n        finally:\n            _set_source(previous_source)\n\n        executed = bool(isinstance(payload, dict) and payload.get("executed"))\n        if executed and isinstance(payload, dict):\n            try:\n                self._last_action_result = self._compact_action_result(payload)\n                self._last_step_summary = self._summarize_step_sequence([payload])\n                self._update_summarized_knowledge_from_step_summary()\n            except Exception:\n                pass\n        detail = (\n            f"fallback_reason={reason}\\n"\n            f"latency_s={latency_s:.3f}\\n"\n            f"selected={choice.compact()}\\n"\n            f"frontier={frontier_text}\\n"\n            f"executed={executed}\\n"\n            f"result={json.dumps(payload, sort_keys=True, default=str)[:1600]}"\n        )\n        self._append_adl_status(transcript_path, detail)\n        if ADL_STDOUT_EVERY_MOVE:\n            print(\n                f"ADL FALLBACK game={self.game_id} move={action_num + 1} "\n                f"reason={reason} selected={choice.key} executed={executed}",\n                flush=True,\n            )\n        return AnalyzerTurnResult(\n            step_executed=executed,\n            retryable_failure=False,\n            reasoning=f"ADL local fallback selected {choice.key} after {reason}.",\n        )\n\n    def analyze(\n        self,\n        state_path: Path,\n        action_num: int,\n        valid_actions: list[str] | None = None,\n        step_env: Callable[[dict[str, Any]], dict[str, Any]] | None = None,\n        transcript_path: Path | None = None,\n        analysis_step: int | None = None,\n        transcript_updated: Callable[[str], None] | None = None,\n        request_timeout_seconds: float | None = None,\n        should_stop: Callable[[], bool] | None = None,\n    ) -> AnalyzerTurnResult | None:\n        started = time.monotonic()\n        self._last_request_error = ""\n        previous_source = _get_source()\n        try:\n            _set_source("model")\n            result = super().analyze(\n                state_path,\n                action_num,\n                valid_actions=valid_actions,\n                step_env=step_env,\n                transcript_path=transcript_path,\n                analysis_step=analysis_step,\n                transcript_updated=transcript_updated,\n                request_timeout_seconds=self._effective_request_timeout(request_timeout_seconds),\n                should_stop=should_stop,\n            )\n        finally:\n            _set_source(previous_source)\n\n        latency = max(0.0, time.monotonic() - started)\n        if result is not None and bool(result.step_executed):\n            return result\n\n        if result is None:\n            reason = "analyzer_exception_or_none"\n        elif bool(result.retryable_failure):\n            reason = "analyzer_request_failure"\n            if self._last_request_error:\n                reason += ":" + self._last_request_error\n        elif bool(getattr(result, "yielded_control", False)):\n            reason = "analyzer_yield_without_action"\n        else:\n            reason = "analyzer_no_action"\n\n        return self._fallback(\n            state_path=state_path,\n            action_num=action_num,\n            valid_actions=valid_actions,\n            step_env=step_env,\n            transcript_path=transcript_path,\n            reason=reason,\n            latency_s=latency,\n            should_stop=should_stop,\n        )\n\n\ndef _game_id_from_session(session: Any) -> str:\n    run = getattr(getattr(session, "game", None), "game_run", None)\n    if run is not None:\n        gid = str(getattr(run, "game_id", "") or "").strip()\n        if gid:\n            return gid\n    game = getattr(session, "game", None)\n    for attr in ("env_name", "game_id", "name"):\n        value = str(getattr(game, attr, "") or "").strip()\n        if value:\n            return value\n    return f"game-{getattr(session, \'game_index\', \'unknown\')}"\n\n\ndef install_duckwadl_adl_v3(\n    benchmark: Any,\n    *,\n    output_dir: str | Path = "/kaggle/working/adl_v3",\n) -> ADLRunMemory:\n    """Patch the loaded Duck solver in-memory without altering the mounted source bundle."""\n    global ADL_SHARED, _PATCHED, _ORIGINAL_MAKE_ANALYZER, _ORIGINAL_EXECUTE_ACTION\n\n    import inference.framework.solver as solver_mod\n\n    with _PATCH_LOCK:\n        ADL_SHARED = ADLRunMemory(output_dir)\n        ADL_SHARED.reset_files()\n\n        if not _PATCHED:\n            _ORIGINAL_MAKE_ANALYZER = solver_mod.HarnessSolver._make_analyzer\n            _ORIGINAL_EXECUTE_ACTION = solver_mod._HarnessGameSession._execute_action\n\n            def _adl_make_analyzer(\n                self: Any,\n                game: Any,\n                index: int,\n                local_server: Any | None = None,\n            ) -> ADLToolAgent:\n                run = getattr(game, "game_run", None)\n                game_id = str(\n                    getattr(run, "game_id", "")\n                    or getattr(game, "env_name", "")\n                    or getattr(game, "game_id", "")\n                    or f"game-{index}"\n                )\n                return ADLToolAgent(\n                    game_id=game_id,\n                    shared_memory=ADL_SHARED,\n                    model=self.model,\n                    timeout=self.analyzer_timeout,\n                    save_request_logs=self.save_request_logs,\n                    api_key=(\n                        getattr(local_server, "api_key", "")\n                        if local_server is not None\n                        else getattr(self, "_local_server_api_key", "")\n                    )\n                    or None,\n                    base_url=(\n                        getattr(local_server, "base_url", "")\n                        if local_server is not None\n                        else getattr(self, "_local_server_base_url", "")\n                    )\n                    or None,\n                    provider="vllm" if local_server is not None else None,\n                )\n\n            def _adl_execute_action(\n                session: Any,\n                action: Any,\n                *,\n                batch_index: int,\n                batch_size: int,\n                generated_tokens: int | None = None,\n                flush_viewer_payload: bool = True,\n            ) -> dict[str, Any]:\n                assert _ORIGINAL_EXECUTE_ACTION is not None\n                before = session.current_frame()\n                action_name = getattr(getattr(action, "id", None), "name", "")\n                action_data = dict(getattr(action, "data", {}) or {})\n                if action_name == "ACTION6":\n                    action_display = (\n                        f"MOUSE(row={int(action_data.get(\'y\', 0))}, "\n                        f"col={int(action_data.get(\'x\', 0))})"\n                    )\n                else:\n                    action_display = to_model_action(action_name)\n                source = _get_source()\n                payload = _ORIGINAL_EXECUTE_ACTION(\n                    session,\n                    action,\n                    batch_index=batch_index,\n                    batch_size=batch_size,\n                    generated_tokens=generated_tokens,\n                    flush_viewer_payload=flush_viewer_payload,\n                )\n                after = session.current_frame()\n                game_id = _game_id_from_session(session)\n                record = ADL_SHARED.record_transition(\n                    game_id=game_id,\n                    move=int(payload.get("action_num") or session.action_count),\n                    before=before,\n                    after=after,\n                    action_display=action_display,\n                    payload=payload,\n                    source=source,\n                )\n\n                try:\n                    with session.transcript_path.open("a", encoding="utf-8") as f:\n                        f.write("[ADL MOVE]\\n")\n                        f.write(\n                            f"game={record.game_id} move={record.move} source={record.source}\\n"\n                            f"action={record.action}\\n"\n                            f"prediction=P(change)={record.predicted_change:.3f} "\n                            f"P(progress)={record.predicted_progress:.3f} "\n                            f"E(diff)={record.predicted_diff_ratio:.4f}\\n"\n                            f"observation=changed={int(record.board_changed)} "\n                            f"diff_cells={record.diff_cells} diff_ratio={record.diff_ratio:.4f} "\n                            f"progress={record.progress_delta} reward={record.reward:.4f}\\n"\n                            f"prediction_error={record.prediction_error:.4f}\\n\\n"\n                        )\n                except OSError:\n                    pass\n\n                if ADL_STDOUT_EVERY_MOVE:\n                    print(\n                        f"ADL MOVE game={record.game_id} move={record.move} "\n                        f"source={record.source} action={record.action} "\n                        f"delta={record.diff_cells} ratio={record.diff_ratio:.4f} "\n                        f"progress={record.progress_delta} pred_err={record.prediction_error:.3f}",\n                        flush=True,\n                    )\n                return payload\n\n            solver_mod.HarnessSolver._make_analyzer = _adl_make_analyzer\n            solver_mod._HarnessGameSession._execute_action = _adl_execute_action\n            _PATCHED = True\n\n        solver = getattr(benchmark, "solver", None)\n        if solver is None:\n            raise RuntimeError("Loaded benchmark has no solver to patch.")\n\n        # The local analyzer can still have its own configured timeout; ADL hard-caps\n        # each request independently so a single 60–120 second stall cannot dominate.\n        solver.analyzer_factory = None\n        label = str(getattr(solver, "label", "HarnessSolver") or "HarnessSolver")\n        if "ADL-v3" not in label:\n            solver.label = f"{label}-DuckWADL-ADL-v3"\n\n        print(\n            "DuckWADL ADL v3 installed: "\n            f"hard_timeout={ADL_ANALYZER_HARD_TIMEOUT_S}s "\n            f"history_turns={ADL_HISTORY_TURNS} "\n            f"tool_steps={ADL_TOOL_STEPS} "\n            f"stdout_every_move={ADL_STDOUT_EVERY_MOVE}",\n            flush=True,\n        )\n        return ADL_SHARED\n\n\ndef finalize_duckwadl_adl_v3(benchmark: Any) -> dict[str, Any]:\n    report = ADL_SHARED.finalize_from_benchmark(benchmark)\n    totals = report.get("totals", {})\n    print(\n        "ADL FINAL "\n        f"games={totals.get(\'games_seen\', 0)} "\n        f"moves={totals.get(\'moves_recorded\', 0)} "\n        f"positive={totals.get(\'positive_traces\', 0)} "\n        f"stalled={totals.get(\'stalled_negative_traces\', 0)} "\n        f"analyzer_failures={totals.get(\'analyzer_failures\', 0)} "\n        f"fallback_moves={totals.get(\'fallback_moves\', 0)}",\n        flush=True,\n    )\n    print("ADL strategy delta:", json.dumps(report.get("strategy_delta", {}), sort_keys=True), flush=True)\n    for item in report.get("guidance", []):\n        print("ADL guidance:", item, flush=True)\n    return report\n'
ADL_MODULE_PATH.write_text(ADL_MODULE_SOURCE, encoding="utf-8")

import importlib.util
spec = importlib.util.spec_from_file_location("duckwadl_adl_v3", ADL_MODULE_PATH)
if spec is None or spec.loader is None:
    raise RuntimeError("Could not load DuckWADL ADL module.")
duckwadl_adl_v3 = importlib.util.module_from_spec(spec)
sys.modules["duckwadl_adl_v3"] = duckwadl_adl_v3
spec.loader.exec_module(duckwadl_adl_v3)

print(f"DuckWADL: ADL module written and loaded from {ADL_MODULE_PATH}")


## 6. ADL self-test — difference detector, candidate frontier, persistence


In [ ]:
from inference.agent.runtime_state import Frame

_test_before = Frame(
    grid=((0, 0, 0, 0), (0, 1, 1, 0), (0, 0, 0, 0)),
    step=0,
    level=1,
)
_test_after = Frame(
    grid=((0, 0, 0, 0), (0, 1, 2, 0), (0, 0, 0, 0)),
    step=1,
    level=1,
)

_test_delta = duckwadl_adl_v3.compare_frames(_test_before, _test_after)
assert _test_delta.changed_cells == 1
assert _test_delta.changed_ratio > 0

_test_memory = duckwadl_adl_v3.ADLRunMemory(WORKING_DIR / "adl_v3_selftest")
_test_memory.reset_files()
choice, frontier = _test_memory.choose_fallback(
    "selftest",
    _test_before,
    ["UP", "DOWN", "LEFT", "RIGHT", "MOUSE"],
)
assert choice is not None
assert frontier
print("DuckWADL: ADL self-test passed.")
print("DuckWADL: sample delta:", _test_delta.compact())
print("DuckWADL: sample frontier:")
for item in frontier:
    print(" ", item.compact())


## 7. Load the deployed benchmark and install ADL into the real Duck solver


In [ ]:
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

adl_memory = duckwadl_adl_v3.install_duckwadl_adl_v3(
    bm,
    output_dir=WORKING_DIR / "adl_v3",
)

print("DuckWADL: solver=", bm.solver)
print("DuckWADL: patched solver label=", getattr(bm.solver, "label", ""))
print("DuckWADL: analyzer timeout configured=", getattr(bm.solver, "analyzer_timeout", None))
print("DuckWADL: concurrency preserved=", getattr(bm.solver, "concurrency", None))


## 8. Competition game construction

A real Kaggle competition rerun uses the live competition Arcade gateway.  
An interactive **Save & Run All** uses the bundled public environment files offline.

The ADL layer is identical in both modes.


In [ ]:
def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]

def _offline_games(env_dir: str):
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]

def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


## 9. Run the actual benchmark


In [ ]:
print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text(
    (BUNDLE_DIR / "git_status.txt").read_text()
)

os.environ.setdefault(
    "RECORDINGS_DIR",
    str(WORKING_DIR / "server_recording"),
)

if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    competition_env_files = str(
        Path(
            "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
        ).parent
        / "environment_files"
    )
    bm.games = _offline_games(competition_env_files)

bm.n_passes = 1
bm.game_weights = None

soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(
            seconds=budget - min(600.0, budget / 2)
        )

print(
    f"DuckWADL: launching {len(bm.games)} games | "
    f"passes={bm.n_passes} | "
    f"concurrency={getattr(bm.solver, 'concurrency', None)} | "
    f"ADL hard request timeout={os.environ['ADL_ANALYZER_HARD_TIMEOUT_S']}s",
    flush=True,
)

run_error = None
try:
    await bm.run(
        soft_end_time=soft_end,
        runtime_environment=target,
        minimal_diagnostics=TRUE_SUBMISSION,
    )
except Exception as exc:
    run_error = exc
    print(f"DuckWADL: benchmark raised {type(exc).__name__}: {exc}", flush=True)
finally:
    # Finalize ADL before the model server is torn down.
    try:
        adl_report = duckwadl_adl_v3.finalize_duckwadl_adl_v3(bm)
    except Exception as adl_exc:
        adl_report = {"error": f"{type(adl_exc).__name__}: {adl_exc}"}
        print("DuckWADL: ADL finalization failed:", adl_report["error"], flush=True)

    if not TRUE_SUBMISSION:
        # Kaggle requires a submission.parquet artifact even for an offline Save & Run.
        import pandas as pd

        submission_path = WORKING_DIR / "submission.parquet"
        if not submission_path.exists():
            pd.DataFrame(
                [["1_0", "1", True, 1]],
                columns=["row_id", "game_id", "end_of_game", "score"],
            ).to_parquet(submission_path, index=False)

    teardown_commands = json.loads(
        (BUNDLE_DIR / "teardown_commands.json").read_text()
    )
    for command in teardown_commands:
        print(f"DuckWADL: teardown command: {command}", flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if run_error is not None:
    raise run_error


## 10. Validate canonical Kaggle submission outputs


In [ ]:
SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
if not SUBMISSION_PATH.is_file():
    raise RuntimeError(
        "Canonical Kaggle output missing: /kaggle/working/submission.parquet"
    )

import pandas as pd

submission_df = pd.read_parquet(SUBMISSION_PATH)
required_columns = ["row_id", "game_id", "end_of_game", "score"]
missing_columns = [c for c in required_columns if c not in submission_df.columns]
if missing_columns:
    raise RuntimeError(
        f"submission.parquet is missing required columns: {missing_columns}; "
        f"actual columns={list(submission_df.columns)}"
    )
if submission_df.empty:
    raise RuntimeError("submission.parquet contains zero rows.")
if submission_df["row_id"].isna().any():
    raise RuntimeError("submission.parquet contains null row_id values.")
if submission_df["game_id"].isna().any():
    raise RuntimeError("submission.parquet contains null game_id values.")

if TRUE_SUBMISSION:
    is_offline_dummy = (
        len(submission_df) == 1
        and str(submission_df.iloc[0]["row_id"]) == "1_0"
        and str(submission_df.iloc[0]["game_id"]) == "1"
        and bool(submission_df.iloc[0]["end_of_game"])
        and float(submission_df.iloc[0]["score"]) == 1.0
    )
    if is_offline_dummy:
        raise RuntimeError(
            "True competition rerun produced the offline placeholder submission. "
            "Refusing to present it as a scored run."
        )

ADL_OUTPUTS = [
    WORKING_DIR / "adl_v3" / "adl_transitions.jsonl",
    WORKING_DIR / "adl_v3" / "adl_analyzer_failures.jsonl",
    WORKING_DIR / "adl_v3" / "adl_report.json",
    WORKING_DIR / "adl_v3" / "adl_memory.json",
]
missing_adl = [str(path) for path in ADL_OUTPUTS if not path.is_file()]
if missing_adl:
    raise RuntimeError(f"Missing ADL output artifacts: {missing_adl}")

manifest = {
    "true_submission": bool(TRUE_SUBMISSION),
    "submission_path": str(SUBMISSION_PATH),
    "submission_rows": int(len(submission_df)),
    "submission_columns": list(submission_df.columns),
    "adl_outputs": [str(path) for path in ADL_OUTPUTS],
}
(WORKING_DIR / "duckwadl_output_manifest.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("DuckWADL: canonical submission output validated.")
print(json.dumps(manifest, indent=2, sort_keys=True))


## 11. ADL Success-Difference Replay report


In [ ]:
ADL_REPORT_PATH = WORKING_DIR / "adl_v3" / "adl_report.json"
if not ADL_REPORT_PATH.is_file():
    raise RuntimeError(f"ADL report was not produced: {ADL_REPORT_PATH}")

adl_report = json.loads(ADL_REPORT_PATH.read_text())
print(json.dumps(adl_report["totals"], indent=2, sort_keys=True))
print("\nStrategy delta:")
print(json.dumps(adl_report["strategy_delta"], indent=2, sort_keys=True))
print("\nLearned guidance:")
for line in adl_report["guidance"]:
    print("-", line)

replays = adl_report.get("success_difference_replay", [])
print(f"\nSuccess-Difference Replay pairs: {len(replays)}")
for replay in replays[:20]:
    print(
        replay["positive_game"],
        "vs",
        replay["negative_game"],
        json.dumps(replay["delta"], sort_keys=True),
    )


## 12. Per-game ADL summary


In [ ]:
games = adl_report.get("games", {})
rows = []
for game_id, item in games.items():
    feat = item.get("features", {})
    rows.append(
        {
            "game_id": game_id,
            "final_score": item.get("final_score"),
            "moves": item.get("moves", 0),
            "progress_events": item.get("progress_events", 0),
            "fallback_moves": item.get("fallback_moves", 0),
            "analyzer_failures": item.get("analyzer_failures", 0),
            "change_rate": feat.get("change_rate", 0.0),
            "zero_effect_rate": feat.get("zero_effect_rate", 0.0),
            "unique_state_rate": feat.get("unique_state_rate", 0.0),
            "mean_diff_ratio": feat.get("mean_diff_ratio", 0.0),
        }
    )

rows.sort(
    key=lambda x: (
        -(float(x["final_score"]) if x["final_score"] is not None else -1.0),
        -int(x["progress_events"]),
        int(x["moves"]),
    )
)

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for row in rows:
        print(row)


## 13. Diagnostics viewer


In [ ]:
from html import escape
from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print(
        "No diagnostics.html — real competition submissions use minimal diagnostics. "
        "ADL JSONL/report artifacts remain in /kaggle/working/adl_v3/."
    )


## What changes in the logs

Every actual environment action now prints a compact line similar to:

```text
ADL MOVE game=tn36-... move=20 source=model action=RIGHT delta=17 ratio=0.0312 progress=1 pred_err=0.217
```

When the local analyzer stalls:

```text
ADL FALLBACK game=... move=113 reason=analyzer_request_failure selected=LEFT executed=True
```

The solver therefore **does not sit in the original retry loop repeating the same failed analyzer request**. The fallback action is chosen from the local causal frontier and becomes another ADL transition immediately.

The full transition record contains:
`state_before`, `action`, prediction probabilities, `state_after`, changed-cell count, difference ratio, level/progress delta, prediction error, source (`model` or `fallback`), and terminal state flags.
